In [28]:
##------------- Load modules -----------
import sys
import os
import mercury as mr
import pandas as pd
import re
import numpy as np
from IPython.display import display, HTML

##------------- display aesthetics -----------
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2E}'.format)
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

##------------- load R commands -----------
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [26]:
import os
rootPath = os.path.join(os.path.dirname(os.path.realpath('__file__')))
sys.path.insert(1, rootPath + "/1_src")
import resFunctions as rF

In [2]:
%%R
##------------- source R scripts ----------- 
suppressMessages(library(ggplot2))
suppressMessages(library(magrittr))
suppressMessages(library(dplyr))
source("/lustre/projects/Research_Project-MRC148213/lsl693/scripts/SFARI_developmentalgenomics/0_mercury/1_src/ggtranscriptBase.R")
source("/lustre/projects/Research_Project-MRC148213/lsl693/scripts/SFARI_developmentalgenomics/0_mercury/1_src/shorten_gaps.R")

In [3]:
class MetadataVariables:
    def __init__(self, meta_dir):
        self.metaDir = meta_dir
        # SQANTI classification file 
        self.classFile = self.metaDir + "classfile.csv"
        # number of transcripts per associated gene
        self.numGenes = self.metaDir + "numGenes.csv"
        # output from DTE
        self.GroupDTE = self.metaDir + "filtered_output_whole_group_removinglateprenatal_TEST.csv"
        self.SexDTE = self.metaDir + "filtered_output_whole_sex_removinglateprenatal_TEST.csv"
        # output from DGE
        self.GroupDGE = self.metaDir + "DGEGroup.csv"
        self.SexDGE = self.metaDir + "DGESex.csv"
        # normalised output from DTE and DGE
        self.GroupDTENorm = self.metaDir + "filtered_whole_group_norm.csv"
        self.SexDTENorm = self.metaDir + "filtered_whole_sex_norm.csv"
        self.GroupDGENorm = self.metaDir + "whole_group_gene_norm.csv"
        # gtf for plotting        
        self.gtf = self.metaDir + "knownGenesExons.gtf"
        self.refgtf = self.metaDir + "refExons.gtf"
        # phenotype
        self.phenotype = self.metaDir + "phenotype.csv"

    def read_num_genes(self):
        return pd.read_csv(self.numGenes)

    def read_group_dte(self):
        return pd.read_csv(self.GroupDTE)

    def read_sex_dte(self):
        return pd.read_csv(self.SexDTE)

    def read_phenotype(self):
        return pd.read_csv(self.phenotype, dtype={'sample': 'string', 'group': 'string', 'sex': 'string', 'Weight': 'float64'})

# Example usage:
inputPath = "/lustre/projects/Research_Project-MRC148213/lsl693/scripts/SFARI_developmentalgenomics/0_mercury/0_metadata/"
inputVars = MetadataVariables(inputPath)

# Read CSV files using the class methods
allClassFileNum = inputVars.read_num_genes()
GroupDTE = inputVars.read_group_dte()
SexDTE = inputVars.read_sex_dte()
phenotype = inputVars.read_phenotype()


In [ ]:
# Title and description
app = mr.App(title="Long-read Transcriptome Resource", description="ONT Latest long-read dataset - Mill2023")
mr.Markdown(text="""
# **Welcome to our ONT long-read transcriptome resource!**

Our study leverages both whole and targeted long read transcriptome sequencing data in 47 human cortex samples which span from prenatal to postnatal ages (6 wpc to 95 years). Please refer to Bamford et al. (2024) for more details.
Of note, current resource only generates annotations and plots for known genes. 
""")

In [ ]:
# type gene name
name = mr.Text(label="Type gene name:")

if name.value: 
    
    # if gene name is known or detected
    if name.value in list(allClassFileNum["associated_gene"]):

        mr.Md(f"""## {name.value}""")
        classFileNum = allClassFileNum[allClassFileNum["associated_gene"] == name.value]
        classFile = rF.subset_file(inputVars.classFile, 1, name.value)
        
        # Display markdown of number of transcripts
        AllTranscripts = classFileNum["totalN"].values[0]
        NovelTranscripts = classFileNum["novelN"].values[0]
        mr.Md(f"Total number of transcripts: {AllTranscripts}")
        mr.Md(f"Number of novel transcripts: {NovelTranscripts}")
        
        # Downstream saved variables
        GroupDTETrans = list(set(list(classFile["isoform"])).intersection(list(GroupDTE["isoform"])))
        SexDTETrans = list(set(list(classFile["isoform"])).intersection(list(SexDTE["isoform"])))
        DTE = GroupDTE[GroupDTE["isoform"].isin(GroupDTETrans)].sort_values(by = "padj")
        DTESex = SexDTE[SexDTE["isoform"].isin(SexDTETrans)].sort_values(by = "padj")
    
    else:
        mr.Md(f"#### {name.value} is not detected in our dataset.")
        mr.Md(f"Please check the spelling of input gene or insert another gene of interest.")
        

In [ ]:
_ = mr.Note(text="Differential gene expression (DGE)")
# checkbox gene expression plot
DGEButton = mr.Checkbox(value=False, label="Plot gene expression", url_key="flag")

if name.value and DGEButton.value:
    
    mr.Markdown(text="""### Gene expression""")
    if len(rF.subset_file(inputVars.GroupDGE, 0, name.value)) > 0: 
        mr.Md(f"Significant changes in gene expression across development")
    else:
        mr.Md(f"No significant changes in gene expression across development")
    
    if len(rF.subset_file(inputVars.SexDGE, 0, name.value)) > 0: 
        mr.Md(f"Significant changes in gene expression with sex")
    else:
        mr.Md(f"No significant changes in gene expression with sex")
    
    genePlot = rF.plot_expression(rF.subset_file(inputVars.GroupDGENorm, 1, name.value))
    print(genePlot)

else:
    pass

In [ ]:
## ***************** 1. Tabulate 
_ = mr.Note(text="Differential transcript expression (DTE)")
# checkbox transcript table output
DTEButton = mr.Checkbox(value=False, label="Show table", url_key="flag")
    
if name.value and DTEButton.value:
    mr.Markdown(text="""### Differentially expressed transcripts""")
    
    # Display markdown of number of DTE 
    mr.Md(f"Number of differentially expressed transcripts across development: {len(GroupDTETrans)}")
    mr.Md(f"Number of differentially expressed transcripts by sex: {len(SexDTETrans)}")
    
    # merge classFile and output from DTE
    dat1 = classFile[classFile["isoform"].isin(GroupDTETrans)].set_index("isoform")
    dat2 = GroupDTE[GroupDTE["isoform"].isin(GroupDTETrans)].set_index("isoform")[["log2FoldChange","pvalue","padj"]]
    x = dat2.join(dat1, how='left') 
    if len(x) > 0:
        display(HTML("<div style='height: 200px'>" + x.style.render() + "</div>"))
    
else:
    pass

In [ ]:
## ***************** 2. Plot
DTEPlotButton = mr.Checkbox(value=False, label="Plot transcript expression", url_key="flag")

mergedGroup = pd.DataFrame()
mergedSex = pd.DataFrame()
if name.value and DTEPlotButton.value:
    
    selectedGroup = mr.Select(label="across development: pre-natal vs post-natal", choices=list(DTE["isoform"]))
    selectedSex = mr.Select(label="across sex: female vs male", choices=list(DTESex["isoform"]))

    if selectedGroup.value is not None:
        mr.Md(f"""###{selectedGroup.value}""")
        tab = rF.transcript_output(classFile, phenotype, GroupDTE, inputVars.GroupDTENorm, selectedGroup.value)
        
        mr.Md(f"""#### Transcript structure""")
        #merged = plot_structure("ONT21_1927_4635","APP")
        mergedGroup = rF.plot_structure(inputVars.gtf, inputVars.refgtf, selectedGroup.value, name.value)
    
    if selectedSex.value is not None:
        mr.Md(f"""###{selectedSex.value}""")
        tab = rF.transcript_output(classFile, phenotype, SexDTE, inputVars.SexDTENorm, selectedSex.value)
        
        mr.Md(f"""#### Transcript structure""")
        mergedSex = rF.plot_structure(inputVars.gtf, inputVars.refgtf, selectedSex.value, name.value)
        
else:
    pass

In [ ]:
%%R -i mergedGroup 
if (nrow(mergedGroup) > 0) {plot_simply(mergedGroup)}

In [ ]:
%%R -i mergedSex

if (nrow(mergedSex) > 0) {plot_simply(mergedSex)}